In [ ]:
#| default_exp vyakarana

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import re
from functools import cache
from difflib import SequenceMatcher
from ganapati.lemma import to_slp1
from ganapati.sanskrit import strip_vedic
from ganapati.text import is_word

Vyākaraṇa is Sanskrit grammar, and its foundation is Pāṇini's Aṣṭādhyāyī, about 4,000 rules (sūtras) numbered by book, chapter and rule, such as 3.1.68. vidyut includes a program that applies these rules step by step, and this module uses it to show how a written word is derived. The second section matches a word-by-word analysis of a line to the line's written words, and the third names the sandhi rule at the join of two words.

## Specs and derivations

A pada is an inflected word. To ask vidyut for one, you describe it in a single string called a spec. A spec is written in SLP1 with its fields separated by `|`, and the first field says what kind of word it is:

    s|<stem>|<liṅga>|<vibhakti>|<vacana>            a noun or adjective from a plain stem (a compound's stem written joined)
    a|<text>                                        an indeclinable (avyaya)
    k|<dhātu>|<kṛt>|<liṅga>|<vibhakti>|<vacana>     a noun from a root and a kṛt suffix, such as a participle
    ka|<dhātu>|<kṛt>                                an indeclinable kṛt form (ktvā, lyap, tumun)
    t|<dhātu>|<prayoga>|<lakāra>|<puruṣa>|<vacana>[|P|A]  a finite verb; P or A asks for active or middle endings

Liṅga is gender, vibhakti case, vacana number, prayoga voice, lakāra tense or mood, and puruṣa person, each given by vidyut's SLP1 name. `s|rAma|puM|praTamA|eka` is rāmaḥ, masculine nominative singular. A dhātu (verbal root) is written `<aupadeśika>,<gaṇa>,<antargaṇa>,<sanādi+…>,<upasarga+…>`: the root as the Dhātupāṭha lists it, its verb class, its sub-class, any suffixes that make a new root such as the causative, and any prefixes, with unused fields left empty. `t|BU,BvAdi,,,|kartari|la~w|praTama|eka|P` is bhavati.

`pada` turns a spec into a vidyut `Pada`, its noun stem made by `stem`, which gives a feminine in -ā or -ī its feminine suffix so sītā declines as sītāyai, not sītaiḥ, and `derive` returns its derivation, a vidyut `Prakriya`. When you also pass the written word as `surf`, `derive` returns only a derivation that ends in that word, or None, so a wrong spec gives nothing rather than some other word. `derives(spec, surf)` asks the same question and returns True or False. `steps` lists a derivation's steps as `(source, rule, result)`, each rule at most once while the form stays the same, with the source `A` for the Aṣṭādhyāyī, `D` for the Dhātupāṭha, `U` for the Uṇādipāṭha and `V` for a vārttika. `norm` prepares a form for comparison with a derivation by removing accents and writing a final anusvāra as m.

To describe a spec in words, `spec_words` gives its grammar in English (given the written form, a doer-voice verb reads middle when only the ātmanepada endings reach it, as kurute), `spec_lemma` gives the headword it stands on, and `spec_root` gives the root with its prefixes, such as `pra+Ap`. `KRT` and `GRAM_EN` hold the English names behind `spec_words`.

vidyut is imported inside the functions that need it, and only `vidyut.prakriya` is loaded, never the kosha or the segmenter. `vy` holds the one `Vyakarana` object, set to record every step.

In [ ]:
#| export
@cache
def vy():
    "Return the shared vidyut `Vyakarana`, which records each step of a derivation."
    from vidyut.prakriya import Vyakarana
    return Vyakarana(log_steps=True)

In [ ]:
#| export
def _dhatu(f):
    from vidyut.prakriya import Dhatu, Gana, Antargana, Sanadi
    aup, g, ag, sn, px = (f.split(',') + [''] * 5)[:5]
    return Dhatu.mula(aup, Gana.from_string(g), antargana=Antargana.from_string(ag) if ag else None,
                      prefixes=px.split('+') if px else None, sanadi=[Sanadi.from_string(x) for x in sn.split('+')] if sn else None)

In [ ]:
#| export
_NOT_NYAP = frozenset('lakzmI tantrI tarI avI starI'.split())   # uṇādi feminines in -ī with no feminine suffix

def stem(t, g):
    "Return the `Pratipadika` of stem `t` in gender `g`, a feminine in -ā or -ī carrying its feminine suffix (sītā, nadī) so vidyut declines it as one."
    from vidyut.prakriya import Pratipadika
    return Pratipadika.nyap(t) if g == 'strI' and len(t) > 2 and t[-1] in 'AI' and t not in _NOT_NYAP else Pratipadika.basic(t)

In [ ]:
#| export
def pada(spec):
    "Turn a spec into a vidyut `Pada`, or None if the spec cannot be read."
    from vidyut.prakriya import Pada, Pratipadika, Krt, Linga, Vibhakti, Vacana, Prayoga, Lakara, Purusha, DhatuPada
    f = (spec or '').split('|')
    try:
        match f[0]:
            case 's': return Pada.Subanta(stem(f[1], f[2]), Linga.from_string(f[2]), Vibhakti.from_string(f[3]), Vacana.from_string(f[4]))
            case 'a': return Pada.make_avyaya(Pratipadika.basic(f[1], is_avyaya=True))
            case 'k': return Pada.Subanta(Pratipadika.krdanta(_dhatu(f[1]), Krt.from_string(f[2])), Linga.from_string(f[3]), Vibhakti.from_string(f[4]), Vacana.from_string(f[5]))
            case 'ka': return Pada.make_avyaya(Pratipadika.krdanta(_dhatu(f[1]), Krt.from_string(f[2])))
            case 't': return Pada.Tinanta(_dhatu(f[1]), Prayoga.from_string(f[2]), Lakara.from_string(f[3]), Purusha.from_string(f[4]), Vacana.from_string(f[5]),
                                          dhatu_pada=dict(P=DhatuPada.Parasmaipada, A=DhatuPada.Atmanepada).get(f[6] if len(f) > 6 else ''))
    except (ValueError, IndexError, TypeError): return None

In [ ]:
#| export
def norm(slp):
    "Prepare an SLP1 form for comparison with a derivation by removing accents and marks and writing a final anusvāra as m."
    t = re.sub(r"[^A-Za-z']", '', slp or '').replace("'", '')
    return t[:-1] + 'm' if t.endswith('M') else t

In [ ]:
#| export
def derive(spec, surf=None):
    "Derive the word a spec names, returning the derivation that ends in `surf` (SLP1) when given, else the first, or None."
    if not (p := pada(spec)): return None
    try: ps = vy().derive(p)
    except Exception: return None
    if surf is None: return ps[0] if ps else None
    return next((x for x in ps if norm(x.text) == norm(surf)), None)

In [ ]:
#| export
def derives(spec, surf): return derive(spec, surf) is not None

In [ ]:
#| export
def steps(p):
    "List a derivation's steps as `(source, rule, result)`, the source being `A` (Aṣṭādhyāyī), `D` (Dhātupāṭha), `U` (Uṇādipāṭha), `V` (vārttika) or another source's own name."
    src = {'azwADyAyI': 'A', 'DAtupAWaH': 'D', 'uRAdipAWaH': 'U', 'vArttikAH': 'V'}
    out, seen = [], set()
    for s in (p.history if p else []):
        x = (src.get(str(s.source), str(s.source)), s.code, ' '.join(r for r in s.result if r))
        if not out or x[2].replace(' ', '') != out[-1][2].replace(' ', ''): seen = set()
        elif x[:2] in seen: continue   # vidyut can apply a naming sūtra twice in one tier (rāmam: 1.1.73, 1.1.43 under 4.1.2)
        seen.add(x[:2]); out.append(x)
    return out

In [ ]:
#| export
KRT = dict(kta='past participle', ktavatu='past active participle', ktvA='absolutive', lyap='absolutive', tumun='infinitive', Satf='present participle',
           SAnac='present middle participle', SAnan='present middle participle', kAnac='perfect middle participle', kvasu='perfect participle',
           tavya='gerundive', tavyat='gerundive', anIyar='gerundive', yat='gerundive', Ryat='gerundive', kyap='gerundive', kvip='root noun',
           tfc='agent noun', Rvul='agent noun', lyuw='action noun', GaY='action noun', ktin='action noun', ac='noun of action or agent', ka='noun of agent')

In [ ]:
#| export
GRAM_EN = {'puM': 'masculine', 'strI': 'feminine', 'napuMsaka': 'neuter', 'praTamA': 'nominative', 'dvitIyA': 'accusative', 'tftIyA': 'instrumental',
           'caturTI': 'dative', 'paYcamI': 'ablative', 'zazWI': 'genitive', 'saptamI': 'locative', 'samboDanam': 'vocative', 'eka': 'singular', 'dvi': 'dual',
           'bahu': 'plural', 'la~w': 'present', 'li~w': 'perfect', 'lu~w': 'periphrastic future', 'lf~w': 'future', 'le~w': 'subjunctive', 'lo~w': 'imperative',
           'la~N': 'imperfect', 'viDili~N': 'optative', 'ASIrli~N': 'benedictive', 'lu~N': 'aorist', 'lf~N': 'conditional', 'praTama': 'third person',
           'maDyama': 'second person', 'uttama': 'first person', 'kartari': 'active', 'karmaRi': 'passive', 'BAve': 'impersonal'}

In [ ]:
#| export
@cache
def _made(f, krt=''):
    "What vidyut makes of a dhātu field alone (its root, its upasargas set apart) or with a kṛt (the kṛdanta's stem): `prApta`, `naS`."
    from vidyut.prakriya import Pratipadika, Krt
    try: ps = vy().derive(Pratipadika.krdanta(_dhatu(f), Krt.from_string(krt)) if krt else _dhatu(f))
    except Exception: return ''
    return ps[0].text if ps else ''

In [ ]:
#| export
def spec_lemma(spec):
    "Return the SLP1 headword a spec stands on: the stem, the indeclinable, a kṛt form's stem (`prApta`), or a verb's root with its prefixes (`pra+Ap`)."
    f = (spec or '').split('|')
    if f[0] in ('s', 'a'): return f[1] if len(f) > 1 else ''
    if f[0] in ('k', 'ka') and len(f) > 2: return _made(f[1], f[2])
    return spec_root(spec) if f[0] == 't' else ''

In [ ]:
#| export
def spec_root(spec):
    "Return the root of a kṛt form or a verb with its prefixes (`pra+Ap`), or '' for a noun from a plain stem."
    f = (spec or '').split('|')
    if f[0] not in ('k', 'ka', 't') or len(f) < 2: return ''
    aup, g, ag, sn, px = (f[1].split(',') + [''] * 5)[:5]
    return '+'.join([*filter(None, px.split('+')), _made(','.join([aup, g, ag, sn, '']))])

In [ ]:
#| export
def _voice(spec, surf):
    "Name a finite verb's voice: 'middle' when only the ātmanepada endings reach `surf`, else 'active' (kartari covers both)."
    f = spec.split('|')
    if f[2] != 'kartari' or not surf or len(f) > 6: return GRAM_EN.get(f[2], f[2])
    return 'middle' if derives(spec + '|A', surf) and not derives(spec + '|P', surf) else 'active'

In [ ]:
#| export
def spec_words(spec, surf=None):
    "Describe a spec's grammar in English words: the kṛt suffix's meaning if there is one, then case, number and gender, or tense, person, number and voice."
    f = (spec or '').split('|')
    match f[0]:
        case 's': return [GRAM_EN.get(x, x) for x in (f[3], f[4], f[2])]
        case 'a': return ['indeclinable']
        case 'k': return [KRT.get(f[2].replace('~', ''), f[2]), *(GRAM_EN.get(x, x) for x in (f[4], f[5], f[3]))]
        case 'ka': return [KRT.get(f[2].replace('~', ''), f[2]), 'indeclinable']
        case 't': return [*(GRAM_EN.get(x, x) for x in (f[3], f[4], f[5])), _voice(spec, surf) if len(f) > 5 else GRAM_EN.get(f[2], f[2])]
    return []

The checks below use two words with well-known derivations. Deriving रामः applies rules 4.1.2, 8.2.66 and 8.3.15, and deriving विजयते applies 3.4.78, 3.1.68, 7.3.84 and 6.1.78. The spec for rāmaḥ checked against the written form rāmeṇa gives None.

In [ ]:
def codes(spec, surf): return [c for s, c, _ in steps(derive(spec, surf)) if s == 'A']
assert derive('s|rAma|puM|praTamA|eka', 'rAmaH').text == 'rAmaH'
assert {'4.1.2', '8.2.66', '8.3.15'} <= set(codes('s|rAma|puM|praTamA|eka', 'rAmaH'))
assert {'3.4.78', '3.1.68', '7.3.84', '6.1.78'} <= set(codes('t|ji\\,BvAdi,,,vi|kartari|la~w|praTama|eka', 'vijayate'))
assert derive('s|rAma|puM|praTamA|eka', 'rAmeRa') is None
assert derive('t|BU,BvAdi,,,|kartari|la~w|praTama|eka|P').text == 'Bavati'
assert spec_words('s|rAma|puM|tftIyA|eka') == ['instrumental', 'singular', 'masculine']
assert spec_lemma('k|A\\px~,svAdi,,,pra|kta|puM|dvitIyA|eka') == 'prApta' and spec_root('k|A\\px~,svAdi,,,pra|kta|puM|dvitIyA|eka') == 'pra+Ap'
assert norm('rAmaM') == 'rAmam'
assert derive('s|sItA|strI|caturTI|eka', 'sItAyE') and derive('s|nadI|strI|tftIyA|eka', 'nadyA') and derive('s|lakzmI|strI|praTamA|eka', 'lakzmIH')
kf = 't|qukf\\Y,tanAdi,,,|kartari|la~w|praTama|eka'
assert spec_words(kf, 'kurute')[-1] == 'middle' and spec_words(kf, 'karoti')[-1] == 'active' and spec_words(kf)[-1] == 'active'
cs = [c for _, c, _ in steps(derive('s|rAma|puM|dvitIyA|eka', 'rAmam'))]
assert cs.count('1.1.73') == 1

## Placing a word-by-word analysis on the written line

A word-by-word analysis lists a line's words in reading order with sandhi undone, so it has more, shorter entries than the line has written tokens. स कालेनेह is two written tokens and three entries: सः, कालेन, इह. `align` works out which entries belong to which token. It takes the tokens and a list of entries, each an object with the word in `.w`, such as the records from `etym_entries`, and returns one list of entry positions per token.

The match keeps the order of both lists. A token may take up to four consecutive entries, and they fit when their joined sandhi skeleton is similar enough to the token's: a difflib similarity ratio of at least 0.72, scored by the token's length. An entry that no token fits is dropped at a cost of 0.5. That cost was tuned on Gītā 4.2. Any lower and कालेनेह loses इह; any higher and विदुः takes a सः that the analysis added but the line does not have.

`sandhi_skeleton` is what makes the comparison tolerant of sandhi. It removes marks, drops a final s, r, ḥ, m or ṃ, and makes long vowels short, since a word's final sound and its vowel length are what sandhi most often changes.

In [ ]:
#| export
def sandhi_skeleton(slp):
    "Reduce an SLP1 word to a form that survives sandhi: marks removed, a final s, r, H, m or M dropped, and long vowels made short."
    t = re.sub(r'[^A-Za-z]', '', slp)
    t = re.sub(r'[srHmM]$', '', t)
    return t.translate(str.maketrans('AIUFX', 'aiufx'))

In [ ]:
#| export
def align(tk, es):
    "Match a line's written tokens to its word-by-word entries in order, returning the list of entry positions each token takes."
    T = [sandhi_skeleton(to_slp1(strip_vedic(t))) if is_word(t) else '' for t in tk]
    E = [sandhi_skeleton(to_slp1(e.w)) for e in es]
    n, m = len(T), len(E)
    NEG = -1e9
    dp = [[NEG] * (m + 1) for _ in range(n + 1)]; bk = [[None] * (m + 1) for _ in range(n + 1)]
    dp[0][0] = 0
    for i in range(n + 1):
        for j in range(m + 1):
            if dp[i][j] == NEG: continue
            if j < m and dp[i][j] - .5 > dp[i][j + 1]: dp[i][j + 1], bk[i][j + 1] = dp[i][j] - .5, (i, j, None)   # drop an entry: dear enough that कालेनेह keeps इह, cheap enough that विदुः does not take a सः the line lacks
            if i == n: continue
            if dp[i][j] > dp[i + 1][j]: dp[i + 1][j], bk[i + 1][j] = dp[i][j], (i, j, ())   # a token left without entries
            if not T[i]: continue
            for k in range(j + 1, min(m, j + 4) + 1):
                cat = ''.join(E[j:k])
                r = SequenceMatcher(None, T[i], cat, autojunk=False).ratio()
                if r < .72: continue
                sc = dp[i][j] + r * len(T[i])
                if sc > dp[i + 1][k]: dp[i + 1][k], bk[i + 1][k] = sc, (i, j, tuple(range(j, k)))
    out, i, j = [[] for _ in range(n)], n, m
    while (i, j) != (0, 0) and bk[i][j]:
        pi, pj, g = bk[i][j]
        if g: out[pi] = list(g)
        i, j = pi, pj
    return out

In [ ]:
from collections import namedtuple

In [ ]:
E = namedtuple('E', 'w')
tk = 'स कालेनेह महता योगो नष्टः परन्तप'.split()
es = [E(x) for x in ('सः', 'कालेन', 'इह', 'महता', 'योगः', 'नष्टः', 'परन्तप')]
assert align(tk, es) == [[0], [1, 2], [3], [4], [5], [6]]
assert sandhi_skeleton('rAmaH') == 'rama'

## The sandhi at a join

Sandhi is the set of sound changes where two words meet: ca and eva are written caiva, and rāmaḥ before gacchati becomes rāmo. `join_rule` takes the word before the join and the word after, both in SLP1 with sandhi undone, and names the rule that joins them. It returns `(id, name, sūtra)` from `JOINS`, such as `('vrddhi', 'vṛddhi', '6.1.88')`, or None for a join that no class fits. `JOINS` describes 25 classes that cover the joins of classical verse, each with its Aṣṭādhyāyī sūtra and a one-line description, plus a `none` entry for words written apart.

Pass the written form as `written` when you have it. Two pairs of rules can only be told apart that way: whether the y or v of ayādi was kept or dropped (te ime written tayime or ta ime), and whether t before ś became cch (tacchrutvā) or only c.

In [ ]:
#| export
_V = 'aAiIuUfFxXeEoO'

In [ ]:
#| export
_SAV = {'a': 'aA', 'A': 'aA', 'i': 'iI', 'I': 'iI', 'u': 'uU', 'U': 'uU', 'f': 'fFxX', 'F': 'fFxX', 'x': 'fFxX', 'X': 'fFxX'}

In [ ]:
#| export
_VOICED = set('gGjJqQdDbBNYRnmyrlvhaAiIuUfFxXeEoO')

In [ ]:
#| export
JOINS = {
    'dirgha': ('savarṇa dīrgha', '6.1.101', 'like vowels make one long vowel'),
    'guna': ('guṇa', '6.1.87', 'a or ā with i, u, ṛ makes e, o, ar'),
    'vrddhi': ('vṛddhi', '6.1.88', 'a or ā with e, ai, o, au makes ai, au'),
    'yan': ('yaṇ', '6.1.77', 'i, u, ṛ before an unlike vowel become y, v, r'),
    'ayadi': ('ayādi', '6.1.78', 'e, o, ai, au before a vowel become ay, av, āy, āv'),
    'lopa_yv': ('the y or v dropped', '8.3.19', 'the y or v of ay, av at a word’s end dropped before a vowel'),
    'purvarupa': ('pūrvarūpa', '6.1.109', 'a after a final e or o merges into it (ऽ)'),
    'pararupa': ('pararūpa', '6.1.94', 'a prefix’s a before e or o takes their form'),
    'o_visarga': ('-aḥ to -o', '6.1.114', '-aḥ before a voiced consonant becomes -o'),
    'o_a': ('-aḥ to -o', '6.1.113', '-aḥ before a becomes -o, and the a merges (ऽ)'),
    'visarga_lopa': ('the visarga dropped', '8.3.22', '-āḥ before a voiced consonant loses its ḥ'),
    'visarga_y': ('the visarga dropped', '8.3.19', '-aḥ or -āḥ before a vowel loses its ḥ'),
    'visarga_r': ('visarga to r', '8.2.66', 'ḥ after another vowel before a voiced sound becomes r'),
    'visarga_s': ('visarga to s, ś, ṣ', '8.3.34', 'ḥ before c, ch, ṭ, ṭh, t, th becomes ś, ṣ, s'),
    'anusvara': ('anusvāra', '8.3.23', 'a final m before a consonant becomes ṃ'),
    'jastva': ('jaśtva', '8.2.39', 'a final stop becomes voiced before a voiced sound'),
    'nasal': ('a nasal for a stop', '8.4.45', 'a final stop before a nasal becomes the nasal of its class'),
    'scutva': ('ścutva', '8.4.40', 't or s before c, j, ś becomes c, j, ś'),
    'stutva': ('ṣṭutva', '8.4.41', 't or s before ṭ, ḍ, ṣ becomes ṭ, ḍ, ṣ'),
    'l': ('t to l', '8.4.60', 't before l becomes l'),
    'ch': ('ś to ch', '8.4.63', 'ś after a stop becomes ch'),
    'n_c': ('n before c, t', '8.3.7', 'n before c, ṭ, t becomes ṃś, ṃṣ, ṃs'),
    'nn': ('n doubled', '8.3.32', 'n after a short vowel doubles before a vowel'),
    'cch': ('ch doubled', '6.1.73', 'ch after a short vowel becomes cch'),
    'sa': ('the visarga of saḥ dropped', '6.1.132', 'saḥ and eṣaḥ lose their ḥ before a consonant'),
    'none': ('no join', '', 'written apart'),
}

In [ ]:
#| export
def _last(w): return re.sub(r"[^A-Za-z]", '', w)[-1:] if w else ''

In [ ]:
#| export
def _first(w): return re.sub(r"[^A-Za-z]", '', w)[:1] if w else ''

In [ ]:
#| export
def join_rule(left, right, written=''):
    "Name the sandhi between `left` and `right` (both SLP1, sandhi undone) as `(id, name, sūtra)` from `JOINS`, using the `written` form when given, or None if no class fits."
    L, R = re.sub(r"[^A-Za-z]", '', left or ''), re.sub(r"[^A-Za-z]", '', right or '')
    if not (L and R): return None
    a, b, w = L[-1], R[0], re.sub(r"[^A-Za-z']", '', written or '')
    def out(k): return (k, *JOINS[k][:2])
    if a in _V and b in _V:
        if a in _SAV and b in _SAV[a]: return out('dirgha')
        if a in 'aA' and b in 'iIuUfF': return out('guna')
        if a in 'aA' and b in 'eEoO': return out('vrddhi')
        if a in 'iIuUfF': return out('yan')
        if a in 'eo' and b == 'a': return out('purvarupa')   # at a word's end always (padāntāt, 6.1.109)
        ys = dict(e='ay', o='av', E='Ay', O='Av')[a]
        return out('ayadi') if not w or ys + b in w else out('lopa_yv')
    if a in 'Hs' and len(L) > 1:
        p = L[-2]
        if p == 'a' and b == 'a': return out('o_a')
        if p == 'a' and b in _VOICED and b not in _V: return out('o_visarga')
        if p in 'aA' and b in _V: return out('visarga_y')
        if p in 'aA' and b in _VOICED: return out('visarga_lopa')
        if b in _VOICED: return out('visarga_r')
        if b in 'cCwWtTkKpP': return out('visarga_s') if b in 'cCwWtT' else None
        return None
    if a == 'm' and b not in _V: return out('anusvara')
    if a == 't':
        if b in 'cCjJS': return out('ch') if b == 'S' and w and 'cC' in w else out('scutva')
        if b in 'wWqQz': return out('stutva')
        if b == 'l': return out('l')
        if b in 'nm': return out('nasal')
        if b in _VOICED: return out('jastva')
    if a in 'kwp' and b in _VOICED: return out('nasal') if b in 'nm' else out('jastva')
    if a == 'n':
        if b in 'cCwWtT': return out('n_c')
        if b in _V and len(L) > 1 and L[-2] in 'aiufx': return out('nn')
    if b == 'C' and a in 'aiufx': return out('cch')
    return None

In [ ]:
for l, r, w, k in [('BIma', 'arjuna', 'BImArjuna', '6.1.101'), ('mahA', 'izvAsAH', 'mahezvAsAH', '6.1.87'), ('ca', 'eva', 'cEva', '6.1.88'),
                   ('karmaRi', 'eva', 'karmaRyeva', '6.1.77'), ('karO', 'uBO', 'karAvuBO', '6.1.78'), ('te', 'api', "te'pi", '6.1.109'),
                   ('te', 'ime', 'ta ime', '8.3.19'), ('rAmaH', 'gacCati', 'rAmo gacCati', '6.1.114'), ('rAmaH', 'api', "rAmo'pi", '6.1.113'),
                   ('hariH', 'gacCati', 'harir gacCati', '8.2.66'), ('tam', 'vande', 'taM vande', '8.3.23'), ('vAk', 'ISa', 'vAgISa', '8.2.39'), ('tat', 'ca', 'tacca', '8.4.40')]:
    assert join_rule(l, r, w)[2] == k, (l, r, join_rule(l, r, w))
assert join_rule('', 'iti') is None

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()